# Lichess Game Analysis, 00 — Dataset Feasibility

This notebook checks whether the Lichess dataset is reliable enough for a focused SQL-based analysis.

The goal of this notebook is not to answer every chess question yet. The goal is to understand the dataset structure, check the main game-level metadata, make clear cleaning decisions, and export a clean dataset for the next notebook.

This project is intentionally smaller than my Brazilian E-commerce analysis project, but the focus is different. I want to strengthen my analytical judgment: not only finding *what* happened in the data, but thinking more carefully about *how* and *why* patterns might appear.

Small note: since I'm not highly experienced in chess, I'm being careful not to overinterpret chess-specific variables too early. For this first stage, I'm focusing on patterns that are understandable from the data itself, such as time controls, ratings, game results, termination type, categories, and openings.


## 1. Setup

The project is organized with separate folders for raw and processed data. The raw Kaggle CSV is kept locally and excluded from GitHub because of its size. The processed dataset exported by this notebook will also stay local and will be used by the SQL analysis notebook.


In [1]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

CSV_PATH = RAW_DIR / "lichess_200k.csv"
PROCESSED_FILE = PROCESSED_DIR / "games_clean.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)


## 2. Load Full Game-Level Metadata

The raw dataset contains hundreds of move, clock, and engine-evaluation columns. Those columns may be useful later, but they are too wide for the first SQL analysis stage.

For this notebook, I load the full dataset using only the game-level metadata columns needed for the first analysis stage. This gives me the full number of games while avoiding the unnecessary memory cost of the repeated move/evaluation/clock columns.

The move, clock, and evaluation columns are deferred to a possible later notebook focused on feature engineering.


In [2]:
metadata_cols = [
    "Index", "Index.1",
    "Date", "UTCDate", "UTCTime", "Weekday",
    "White", "WhiteElo", "WhiteRatingDiff",
    "Black", "BlackElo", "BlackRatingDiff",
    "Result", "Termination",
    "TimeControl", "Category",
    "ECO", "Opening"
]

metadata_df = pd.read_csv(
    CSV_PATH,
    usecols=metadata_cols,
    low_memory=False
)

metadata_df.shape


(200000, 18)

In [3]:
metadata_df.head()


,Index,Index.1,Black,BlackElo,BlackRatingDiff,Date,ECO,Opening,Result,Termination,TimeControl,UTCDate,UTCTime,White,WhiteElo,WhiteRatingDiff,Category,Weekday
0,0,124,albertoPlasta,906,13.00,2019.04.30,B15,Caro-Kann Defense,0-1,Normal,300+3,2019.04.30,22:00:24,KingMateo,971,-12.00,Blitz,Tuesday
1,1,494,Luckystriker,1296,28.00,2019.04.30,C50,Italian Game,0-1,Normal,300+0,2019.04.30,22:00:13,G_i_n,1312,-10.00,Blitz,Tuesday
2,2,552,Verascardoso,1761,-13.00,2019.04.30,C41,Philidor Defense #2,1-0,Normal,600+0,2019.04.30,22:00:41,bufone,1653,27.00,Rapid,Tuesday
3,3,588,Stockfish94,2404,8.00,2019.04.30,B06,Modern Defense,0-1,Normal,60+0,2019.04.30,22:00:43,bcn000,2324,-8.00,Bullet,Tuesday
4,4,725,deneme12376,1595,-10.00,2019.04.30,B32,Sicilian Defense: Loewenthal Variation,1-0,Normal,180+0,2019.04.30,22:00:46,alcool50,1614,29.00,Blitz,Tuesday


## 3. Row Grain and Identifier Columns

Before analyzing the data, I need to understand what one row represents.

Based on the dataset structure, each row appears to represent one chess game. The raw dataset also contains two index-like columns, `Index` and `Index.1`. These are not chess variables; they look like export/source index artifacts.

The checks below show that `Index` is unique across the full metadata dataset, while `Index.1` is not unique. For the analysis itself, I will create a clean `game_id` from the current row order. I will also preserve `Index.1` as `source_row_id` only as a reference back to the raw file, not as a unique game identifier.


In [4]:
metadata_df[["Index", "Index.1"]].head(10)


,Index,Index.1
0,0,124
1,1,494
2,2,552
3,3,588
4,4,725
5,5,832
6,6,1004
7,7,1133
8,8,1159
9,9,1683


In [5]:
metadata_df[["Index", "Index.1"]].nunique()


Index      200000
Index.1     25105
dtype: int64

In [6]:
metadata_df[["Index", "Index.1"]].isna().sum()


Index      0
Index.1    0
dtype: int64

In [7]:
# Preserve the more source-like index for traceability, then create a clean analysis ID.
# If the second index is not useful in a future version of the data, this can be dropped safely.
metadata_df = metadata_df.rename(columns={"Index.1": "source_row_id"})
metadata_df = metadata_df.drop(columns=["Index"], errors="ignore")

metadata_df = metadata_df.reset_index(drop=True)
metadata_df.insert(0, "game_id", metadata_df.index + 1)

metadata_df[["game_id", "source_row_id"]].head()


,game_id,source_row_id
0,1,124
1,2,494
2,3,552
3,4,588
4,5,725


## 4. Date and Time Checks

The dataset contains both `Date` and `UTCDate`. Before dropping either column, I want to check whether they carry the same information across the full metadata dataset.

I also want to inspect the date range because weekday patterns can be misleading if the dataset only covers part of a week or a narrow collection window.


In [8]:
(metadata_df["Date"] == metadata_df["UTCDate"]).mean()


np.float64(1.0)

In [9]:
metadata_df["game_date"] = pd.to_datetime(metadata_df["Date"], format="%Y.%m.%d", errors="coerce")

metadata_df["game_date"].agg(["min", "max"])


min   2019-04-30
max   2019-05-31
Name: game_date, dtype: datetime64[us]

In [10]:
metadata_df["game_date"].isna().sum()


np.int64(0)

In [11]:
metadata_df["game_date"].dt.day_name().value_counts()


game_date
Wednesday    32374
Thursday     32100
Friday       31849
Sunday       26496
Saturday     26325
Tuesday      26005
Monday       24851
Name: count, dtype: int64

### Date and Time Decision

Since `Date` and `UTCDate` are identical across the full metadata dataset, I will keep one parsed date column and drop the redundant raw `UTCDate` column before export.

The weekday distribution should be interpreted together with the date range. If the dataset covers a limited time window, weekday differences may reflect the data collection period rather than true chess-playing behavior.


## 5. Data Quality Checks

Before moving into SQL analysis, I need to check whether the main game-level columns are reliable enough to use.

The focus here is the game-level metadata. Missing values in late move/evaluation/clock columns are not part of this section because those columns are deferred to a later feature-engineering notebook. For this notebook, the key question is whether the columns needed for the first SQL analysis stage are complete, consistent, and structurally reliable.


### 5.1 Missing Values in Key Metadata Columns

I first check missing values in the core game-level columns. Missing values in columns such as rating, result, time control, category, opening, or termination could directly affect the analysis.


In [12]:
key_metadata_cols = [
    "game_id", "source_row_id",
    "game_date", "UTCTime", "Weekday",
    "White", "WhiteElo", "WhiteRatingDiff",
    "Black", "BlackElo", "BlackRatingDiff",
    "Result", "Termination",
    "TimeControl", "Category",
    "ECO", "Opening"
]

metadata_df[key_metadata_cols].isna().sum().sort_values(ascending=False)


WhiteRatingDiff    509
BlackRatingDiff    509
game_id              0
UTCTime              0
source_row_id        0
Weekday              0
White                0
WhiteElo             0
game_date            0
Black                0
BlackElo             0
Result               0
Termination          0
TimeControl          0
Category             0
ECO                  0
Opening              0
dtype: int64

In [13]:
(metadata_df[key_metadata_cols].isna().mean() * 100).sort_values(ascending=False).round(3)


WhiteRatingDiff   0.25
BlackRatingDiff   0.25
game_id           0.00
UTCTime           0.00
source_row_id     0.00
Weekday           0.00
White             0.00
WhiteElo          0.00
game_date         0.00
Black             0.00
BlackElo          0.00
Result            0.00
Termination       0.00
TimeControl       0.00
Category          0.00
ECO               0.00
Opening           0.00
dtype: float64

In [14]:
missing_rating_diff_rows = metadata_df.loc[
    metadata_df["WhiteRatingDiff"].isna() | metadata_df["BlackRatingDiff"].isna(),
    [
        "Result", "Termination", "Category",
        "WhiteElo", "BlackElo",
        "WhiteRatingDiff", "BlackRatingDiff"
    ]
]

missing_rating_diff_rows.head(20)


,Result,Termination,Category,WhiteElo,BlackElo,WhiteRatingDiff,BlackRatingDiff
389,1-0,Normal,Blitz,1943,1948,NaN,NaN
677,0-1,Normal,Blitz,1741,1656,NaN,NaN
1718,0-1,Normal,Rapid,1792,2327,NaN,NaN
3217,1-0,Normal,Blitz,1777,1723,NaN,NaN
3883,1-0,Normal,Rapid,1671,1919,NaN,NaN
5120,1-0,Time forfeit,Blitz,1527,1500,NaN,NaN
5751,1-0,Normal,Blitz,1875,1877,NaN,NaN
5916,0-1,Normal,Classical,1934,1689,NaN,NaN
5988,0-1,Normal,Rapid,1683,1914,NaN,NaN
6860,0-1,Normal,Rapid,1086,2023,NaN,NaN


In [15]:
missing_rating_diff_count = missing_rating_diff_rows.shape[0]
missing_rating_diff_pct = missing_rating_diff_count / len(metadata_df) * 100

missing_rating_diff_count, round(missing_rating_diff_pct, 3)


(509, 0.255)

The main game-level metadata is mostly complete. The only missing values appear in `WhiteRatingDiff` and `BlackRatingDiff`, with 509 missing values in each column, representing about 0.255% of the full 200,000-row metadata dataset.

After inspecting the affected rows, the missing rating-difference values do not appear to be explained by draws or unusual termination types. Many of the affected games have normal results and clear winners.

A missing rating change is not the same as a rating change of zero, so these values should not be filled with `0`. The affected rows can still be used for general game-level outcome analysis, but analyses that directly use post-game rating changes should filter or handle these rows explicitly.


### 5.2 Duplicate Records Check

After checking missing values, I check whether the dataset contains duplicated game records. Duplicate rows could distort counts, rates, and outcome patterns.

I use two duplicate checks:

1. Exact duplicates across the full dataframe.
2. Duplicate game-like records based on a smaller set of identifying metadata columns.

The goal is not to prove with absolute certainty that every game is unique, but to check whether there are obvious repeated records that could affect the analysis.


In [16]:
exact_duplicates = metadata_df.duplicated().sum()
exact_duplicates


np.int64(0)

In [17]:
duplicate_check_cols = [
    "White", "Black",
    "Date", "UTCTime",
    "Result", "TimeControl"
]

metadata_duplicates = metadata_df.duplicated(subset=duplicate_check_cols).sum()
metadata_duplicates


np.int64(0)

In [18]:
metadata_df.loc[
    metadata_df.duplicated(subset=duplicate_check_cols, keep=False),
    duplicate_check_cols + ["Opening", "Termination", "Category"]
].sort_values(["Date", "UTCTime", "White", "Black"]).head(20)


,White,Black,Date,UTCTime,Result,TimeControl,Opening,Termination,Category


### 5.3 Categorical Consistency Checks

Next, I inspect the main categorical columns. This helps confirm that categories are not split by inconsistent spelling, casing, or unexpected labels.

This is especially important before using grouped SQL queries, because SQL will treat labels such as `Blitz` and `Blitz ` as different categories.


In [19]:
categorical_cols = ["Result", "Termination", "Category", "Weekday"]

for col in categorical_cols:
    print(f"\n{col.upper()}")
    print(metadata_df[col].value_counts(dropna=False))



RESULT
Result
1-0        100351
0-1         94499
1/2-1/2      5145
*               5
Name: count, dtype: int64

TERMINATION
Termination
Normal              151517
Time forfeit         48466
Rules infraction        12
Abandoned                5
Name: count, dtype: int64

CATEGORY
Category
Blitz        86333
Rapid        45216
Bullet       43253
Classical    25198
Name: count, dtype: int64

WEEKDAY
Weekday
Wednesday    32374
Thursday     32100
Friday       31849
Sunday       26496
Saturday     26325
Tuesday      26005
Monday       24851
Name: count, dtype: int64


In [20]:
metadata_df["TimeControl"].value_counts(dropna=False).head(20)


TimeControl
600+0      35993
60+0       29087
300+0      28437
900+15     22241
180+0      21791
300+3      18536
180+2      14682
120+1       9534
15+0        1496
900+0       1233
30+0         963
300+5        893
1200+10      797
300+8        744
600+5        660
60+1         627
120+0        602
1200+0       541
480+0        523
900+10       473
Name: count, dtype: int64

The categorical columns are mostly consistent, but the full metadata check reveals a small edge case: 5 games have `Result = '*'` and correspond to abandoned games. Since `*` does not identify a White win, Black win, or draw, these games will not receive a `winner` value later.

This is not a major data-quality problem because the affected group is very tiny compared to the full dataset. The decision is to keep these rows in the processed dataset, but to treat `winner` as missing for abandoned games and exclude or handle them separately in win-rate analysis.

`Rules infraction` and `Abandoned` are also very small termination categories, so their percentages should not be interpreted as meaningful patterns.


### 5.4 Numeric Rating Sanity Checks

Even if rating columns have few missing values, they can still contain suspicious ranges or extreme values. I inspect the rating columns to check whether player ratings and post-game rating changes look reasonable.


In [21]:
numeric_cols = ["WhiteElo", "BlackElo", "WhiteRatingDiff", "BlackRatingDiff"]
metadata_df[numeric_cols].describe()


,WhiteElo,BlackElo,WhiteRatingDiff,BlackRatingDiff
count,"200,000.00","200,000.00","199,491.00","199,491.00"
mean,"1,511.80","1,512.33",4.09,3.24
std,322.31,322.46,31.70,29.93
min,800.00,800.00,-592.00,-600.00
25%,"1,275.00","1,276.00",-10.00,-10.00
50%,"1,500.00","1,500.00",2.00,-2.00
75%,"1,732.00","1,732.00",11.00,11.00
max,"3,091.00","3,110.00",692.00,682.00


In [22]:
rating_diff_outliers = metadata_df.assign(
    abs_white_rating_diff=metadata_df["WhiteRatingDiff"].abs(),
    abs_black_rating_diff=metadata_df["BlackRatingDiff"].abs()
)

rating_diff_outliers.sort_values(
    ["abs_white_rating_diff", "abs_black_rating_diff"],
    ascending=False
)[
    [
        "WhiteElo", "BlackElo",
        "WhiteRatingDiff", "BlackRatingDiff",
        "Result", "Termination",
        "Category", "TimeControl",
        "abs_white_rating_diff", "abs_black_rating_diff"
    ]
].head(20)


,WhiteElo,BlackElo,WhiteRatingDiff,BlackRatingDiff,Result,Termination,Category,TimeControl,abs_white_rating_diff,abs_black_rating_diff
28424,1166,2170,692.00,-26.00,1-0,Normal,Rapid,900+0,692.00,26.00
150466,1500,2504,663.00,-42.00,1-0,Normal,Bullet,30+0,663.00,42.00
64208,1500,2425,661.00,-44.00,1-0,Normal,Blitz,180+0,661.00,44.00
67177,1500,2117,600.00,-17.00,1-0,Time forfeit,Bullet,15+0,600.00,17.00
66380,1500,2084,594.00,-14.00,1-0,Time forfeit,Bullet,30+0,594.00,14.00
134264,1500,908,-592.00,14.00,0-1,Normal,Rapid,600+15,592.00,14.00
147631,1500,2077,592.00,-13.00,1-0,Normal,Bullet,30+0,592.00,13.00
69732,1500,2085,582.00,-20.00,1-0,Time forfeit,Bullet,15+0,582.00,20.00
105581,1500,2041,563.00,-16.00,1-0,Normal,Rapid,300+5,563.00,16.00
45581,1500,2037,561.00,-16.00,1-0,Normal,Blitz,180+0,561.00,16.00


Some post-game rating changes may be large. I will not automatically treat large values as errors/outliers, because online rating systems can produce large changes for newer or unstable ratings, especially after unexpected results (For example, a high rating difference between both players).

The main decision is to keep these rows, but to use post-game rating-difference columns carefully. For most SQL analysis, starting ratings such as `WhiteElo`, `BlackElo`, and the pre-game rating gap are more reliable than post-game rating changes.


## 6. Create Basic Analysis Features

Before exporting the processed dataset, I create a few stable game-level features.

These are not final analytical conclusions. They are basic reusable fields that make the SQL notebook cleaner:

- `winner` from `Result`
- `rating_diff_white_minus_black`
- `abs_rating_diff`
- `rating_favorite`
- `initial_seconds` and `increment_seconds` from `TimeControl`

The `winner` feature is expected to be missing for abandoned games where `Result = '*'`, because those rows do not identify a White win, Black win, or draw. More interpretive features, such as upset flags or category-specific conclusions, can still be created and tested inside SQL.


In [23]:
metadata_df["winner"] = metadata_df["Result"].map({
    "1-0": "White",
    "0-1": "Black",
    "1/2-1/2": "Draw"
})

metadata_df["rating_diff_white_minus_black"] = metadata_df["WhiteElo"] - metadata_df["BlackElo"]
metadata_df["abs_rating_diff"] = metadata_df["rating_diff_white_minus_black"].abs()

def get_rating_favorite(diff, threshold=50):
    if diff > threshold:
        return "White favorite"
    elif diff < -threshold:
        return "Black favorite"
    else:
        return "Close rating"

metadata_df["rating_favorite"] = metadata_df["rating_diff_white_minus_black"].apply(get_rating_favorite)

metadata_df[
    [
        "Result", "winner",
        "WhiteElo", "BlackElo",
        "rating_diff_white_minus_black", "abs_rating_diff",
        "rating_favorite",
        "TimeControl"
    ]
].head()


,Result,winner,WhiteElo,BlackElo,rating_diff_white_minus_black,abs_rating_diff,rating_favorite,TimeControl
0,0-1,Black,971,906,65,65,White favorite,300+3
1,0-1,Black,1312,1296,16,16,Close rating,300+0
2,1-0,White,1653,1761,-108,108,Black favorite,600+0
3,0-1,Black,2324,2404,-80,80,Black favorite,60+0
4,1-0,White,1614,1595,19,19,Close rating,180+0


In [24]:
metadata_df[["winner", "rating_favorite"]].isna().sum()


winner             5
rating_favorite    0
dtype: int64

The engineered feature check confirms that `rating_favorite` has no missing values. The `winner` column has 5 missing values, which matches the 5 games where `Result = '*'` and `Termination = 'Abandoned'`.

I will keep these abandoned games in the processed dataset for completeness, but they should be excluded or handled separately in any analysis that depends on knowing the game outcome, such as win rates, rating-favorite performance, or upset analysis.


## 7. Early Feasibility Insights

This section is not the final analysis. It is a quick check to see whether the dataset contains promising patterns worth exploring in SQL.

The main early thread is time control and termination type. I want to check whether faster formats, especially Bullet, are more likely to end by time forfeit.

For safety, I inspect both raw counts and normalized rates. Counts show the sample size behind each group, while normalized rates make categories easier to compare.


In [25]:
termination_counts_by_category = pd.crosstab(
    metadata_df["Category"],
    metadata_df["Termination"]
)

termination_counts_by_category


Termination,Abandoned,Normal,Rules infraction,Time forfeit
Category,,,,
Blitz,0,67554,1,18778
Bullet,0,21758,0,21495
Classical,1,22970,7,2220
Rapid,4,39235,4,5973


In [26]:
termination_rates_by_category = pd.crosstab(
    metadata_df["Category"],
    metadata_df["Termination"],
    normalize="index"
).round(3)

termination_rates_by_category


Termination,Abandoned,Normal,Rules infraction,Time forfeit
Category,,,,
Blitz,0.00,0.78,0.00,0.22
Bullet,0.00,0.50,0.00,0.50
Classical,0.00,0.91,0.00,0.09
Rapid,0.00,0.87,0.00,0.13


In [27]:
winner_counts_by_termination = pd.crosstab(
    metadata_df["Termination"],
    metadata_df["winner"]
)

winner_counts_by_termination


winner,Black,Draw,White
Termination,,,
Normal,70363,4800,76354
Rules infraction,7,0,5
Time forfeit,24129,345,23992


In [28]:
winner_rates_by_termination = pd.crosstab(
    metadata_df["Termination"],
    metadata_df["winner"],
    normalize="index"
).round(3)

winner_rates_by_termination


winner,Black,Draw,White
Termination,,,
Normal,0.46,0.03,0.50
Rules infraction,0.58,0.00,0.42
Time forfeit,0.50,0.01,0.49


The early crosstabs are useful because they suggest a clear SQL direction: faster time controls may change how games end, especially through time forfeits.

The raw counts are important here. `Rules infraction` and `Abandoned` are very small categories, so their percentages should not be treated as meaningful patterns. The stronger and more stable pattern is the difference in time-forfeit rates across Bullet, Blitz, Rapid, and Classical games.

These are still feasibility-level observations. The SQL notebook will test these patterns more cleanly and will connect them to rating advantage, upset rates, and termination type.


## 8. Cleaning Decisions

A few quick notes before moving to SQL:

`Index.1` wasn't a real unique key, so I created a clean `game_id` instead and kept `source_row_id` just for traceability. `Date` and `UTCDate` were identical across all rows so I dropped the duplicate. The 5 abandoned games stay in he dataet but `winner` is null for them. I'll filter those out in any result-based queries. Large rating-change values look like provisional rating noise, so I'll stick to starting Elo rather than post-game diffs for analysis.

## 9. Feasibility Decision

The game-level metadata appears reliable enough to continue with SQL-based analysis.

The dataset has a clear row grain, usable rating columns, interpretable game results, meaningful time-control information, and useful termination labels. The main issues are manageable: raw index artifact columns, a small amount of missing post-game rating-difference data, 5 abandoned games without a defined winner, and some large rating-change values that appear plausible rather than corrupted.

For this project, the first SQL notebook should focus on game-level analysis. The move, clock, and engine-evaluation columns should be treated as a later extension rather than forced into the first analysis stage.


## 10. Export Clean Game-Level Dataset

After completing the feasibility checks, the main game-level metadata appears reliable enough to continue with SQL-based analysis.

The next step is to create a clean game-level dataset that can be reused in the SQL notebook. This processed file contains only the columns needed for the first analysis stage, instead of carrying the full raw dataset with hundreds of move, clock, and evaluation columns.

This export creates a clear separation between the two notebooks:

- `00_dataset_feasibility.ipynb` checks the dataset, selects the relevant game-level columns, and exports a processed dataset.
- `01_sql_game_analysis.ipynb` loads the processed dataset into SQLite and focuses on SQL analysis.

The raw dataset remains excluded from GitHub because of its large size, while the processed file is used locally for analysis.


In [29]:
export_cols = [
    "game_id", "source_row_id",
    "game_date", "UTCTime", "Weekday",
    "White", "WhiteElo", "WhiteRatingDiff",
    "Black", "BlackElo", "BlackRatingDiff",
    "rating_diff_white_minus_black", "abs_rating_diff", "rating_favorite",
    "Result", "winner", "Termination",
    "TimeControl", "Category",
    "ECO", "Opening"
]

games_clean = metadata_df[export_cols].copy()

games_clean.to_csv(PROCESSED_FILE, index=False)

games_clean.shape


(200000, 21)

In [30]:
PROCESSED_FILE.exists()


True

In [31]:
games_clean.head()


,game_id,source_row_id,game_date,UTCTime,Weekday,White,WhiteElo,WhiteRatingDiff,Black,BlackElo,BlackRatingDiff,rating_diff_white_minus_black,abs_rating_diff,rating_favorite,Result,winner,Termination,TimeControl,Category,ECO,Opening
0,1,124,2019-04-30,22:00:24,Tuesday,KingMateo,971,-12.00,albertoPlasta,906,13.00,65,65,White favorite,0-1,Black,Normal,300+3,Blitz,B15,Caro-Kann Defense
1,2,494,2019-04-30,22:00:13,Tuesday,G_i_n,1312,-10.00,Luckystriker,1296,28.00,16,16,Close rating,0-1,Black,Normal,300+0,Blitz,C50,Italian Game
2,3,552,2019-04-30,22:00:41,Tuesday,bufone,1653,27.00,Verascardoso,1761,-13.00,-108,108,Black favorite,1-0,White,Normal,600+0,Rapid,C41,Philidor Defense #2
3,4,588,2019-04-30,22:00:43,Tuesday,bcn000,2324,-8.00,Stockfish94,2404,8.00,-80,80,Black favorite,0-1,Black,Normal,60+0,Bullet,B06,Modern Defense
4,5,725,2019-04-30,22:00:46,Tuesday,alcool50,1614,29.00,deneme12376,1595,-10.00,19,19,Close rating,1-0,White,Normal,180+0,Blitz,B32,Sicilian Defense: Loewenthal Variation


## 11. Next Notebook Plan

The next notebook will load `games_clean.csv` into SQLite and use SQL to answer the main metadata-level analysis questions.

The first SQL analysis thread should start with game category and termination type, because the feasibility checks already suggested a promising pattern around Bullet games and time forfeits.

Planned SQL questions:

* How do game endings differ across Bullet, Blitz, Rapid, and Classical?
* Are time forfeits much more common in faster categories?
* Are time-forfeit wins balanced between White and Black?
* How often does the rating favorite win?
* Does rating-favorite performance differ across game categories?
* Does the size of the rating gap affect how reliable the favorite is?
* Does time across game categories?
* Does the size of the rating gap affect how reliable the favorite is?
* Does time-forfeit pressure make rating advantage less reliable?
* Which metadata signal appears strongest for favorite reliability: rating gap, category, or termination type?

The goal is to move from surface-level results to stronger reasoning about how time pressure, rating gaps, and termination types shape online chess outcomes.
